# TT-09 — AdaBoost: Phát hiện xâm nhập mạng (NSL-KDD)

Trung tâm điều hành an ninh (SOC) cần phân loại kết nối mạng: **bình thường** hay **tấn công**.

**Dữ liệu cần có** (đặt vào `../data/`, xem README mục "Dữ liệu"):
- `KDDTrain+.arff`
- `KDDTest+.arff` (tập test GỐC của NSL-KDD — cố ý chứa loại tấn công KHÔNG có trong train,
  mô phỏng tấn công zero-day)

> ⚠️ **Lưu ý định dạng:** đây là bản **ARFF** (WEKA) của NSL-KDD, khác bản `.txt` gốc ở hai điểm:
> (1) có block header `@attribute`/`@data` cần bỏ qua khi đọc, và (2) cột nhãn `class` đã được
> **gộp sẵn thành nhị phân** (`normal` / `anomaly`) — không còn giữ tên tấn công cụ thể
> (`neptune`, `smurf`, `guess_passwd`...) như bản `.txt`. Vì vậy notebook này **không tách được**
> nhóm tấn công DoS/Probe/R2L/U2R như mô tả tham khảo trong README — EDA chỉ dừng ở mức
> normal vs attack. Nếu cần phân tích chi tiết theo nhóm tấn công, dùng bản `KDDTrain+.txt`/
> `KDDTest+.txt` gốc (có cột `label` cụ thể + `difficulty_level`) thay vì bản ARFF này.

**Nguyên tắc phương pháp (áp dụng bài học từ TT-07):** `KDDTest+.arff` chỉ được dùng **đúng một
lần**, ở bước đánh giá cuối (mục 9) — để đo đúng mức độ mô hình sụt điểm trước tấn công lạ. Mọi lựa
chọn siêu tham số, đường F1 theo n_estimators, và thí nghiệm nhiễu nhãn đều dùng một tập
**validation tách riêng từ `KDDTrain+.arff`**, không đụng tới `KDDTest+.arff`.


In [ ]:
from __future__ import annotations

import time
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                              confusion_matrix, ConfusionMatrixDisplay)
import joblib

RANDOM_STATE = 42

## 1. Đọc dữ liệu ARFF & tạo nhãn nhị phân

File ARFF gồm 2 phần: block `@attribute ...` mô tả 41 đặc trưng + `class`, và block dữ liệu sau
dòng `@data` (thuần CSV, không header). Ta bỏ qua toàn bộ phần khai báo attribute — chỉ cần đọc
đúng thứ tự 42 cột — và parse phần dữ liệu bằng `pandas.read_csv` thông thường.

In [ ]:
COLS_ARFF = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land',
    'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count',
    'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'class',   # 'normal' hoac 'anomaly' -- ban ARFF khong co ten tan cong cu the/difficulty_level
]


def load_nslkdd_arff(path: Path) -> pd.DataFrame:
    # Doc 1 file NSL-KDD dinh dang ARFF: bo qua block '@attribute', doc CSV tu sau '@data'.
    with open(path, 'r') as f:
        lines = f.readlines()
    data_start = next(i for i, l in enumerate(lines) if l.strip().lower() == '@data') + 1
    csv_text = ''.join(lines[data_start:])
    df = pd.read_csv(StringIO(csv_text), header=None, names=COLS_ARFF)
    df['attack'] = (df['class'] != 'normal').astype(int)   # nhan nhi phan dung de huan luyen
    df = df.drop(columns=['class'])
    return df


# Nhieu cong cu upload/dong bo file se tu doi ky tu '+' trong ten file NSL-KDD thanh '_'
# (vi du 'KDDTrain+.arff' -> 'KDDTrain_.arff'). Ham duoi day tu do doan ca hai kieu ten,
# de tranh FileNotFoundError chi vi khac dau '+' / '_'.
TRAIN_NAME_CANDIDATES = ['KDDTrain+.arff', 'KDDTrain_.arff', 'KDDTrain+_.arff']
TEST_NAME_CANDIDATES = ['KDDTest+.arff', 'KDDTest_.arff', 'KDDTest+_.arff']


def _find_file(dir_path: Path, candidates: list[str]) -> Path | None:
    for name in candidates:
        p = dir_path / name
        if p.exists():
            return p
    return None


def find_data_files() -> tuple[Path, Path]:
    for c in [Path('../data'), Path('data'), Path('../../data')]:
        if not c.exists():
            continue
        train_path = _find_file(c, TRAIN_NAME_CANDIDATES)
        test_path = _find_file(c, TEST_NAME_CANDIDATES)
        if train_path is not None and test_path is not None:
            return train_path, test_path
    raise FileNotFoundError(
        "Khong tim thay du KDDTrain+.arff va KDDTest+.arff (hoac bien the ten voi dau '_' thay "
        "vi '+', do cong cu upload doi ten). Dat ca hai file vao thu muc 'data/' cung cap "
        "'notebooks/' (xem README muc 'Du lieu'), giu nguyen ten goc neu co the.")


TRAIN_PATH, TEST_PATH = find_data_files()
DATA_DIR = TRAIN_PATH.parent
print('Dang dung file train:', TRAIN_PATH)
print('Dang dung file test :', TEST_PATH)

train_df = load_nslkdd_arff(TRAIN_PATH)
test_df = load_nslkdd_arff(TEST_PATH)
print('Train:', train_df.shape, '| Test (NSL-KDD goc, co tan cong la):', test_df.shape)
print('Ti le attack (train):', round(train_df['attack'].mean(), 3))
print('Ti le attack (test):', round(test_df['attack'].mean(), 3))

## 2. EDA nhanh: normal vs attack

Bản ARFF không giữ tên tấn công cụ thể nên không tách được nhóm DoS/Probe/R2L/U2R (xem lưu ý ở đầu
notebook). EDA ở đây dừng ở mức so sánh tỉ lệ `normal`/`attack` giữa train và test — đủ để thấy
liệu tỉ lệ lớp có lệch nhiều giữa hai tập không (ảnh hưởng cách đọc kết quả ở mục 9).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
train_df['attack'].value_counts().rename({0: 'normal', 1: 'attack'}).plot(
    kind='bar', ax=ax[0], title='Train: normal vs attack')
test_df['attack'].value_counts().rename({0: 'normal', 1: 'attack'}).plot(
    kind='bar', ax=ax[1], title='Test goc: normal vs attack')
plt.tight_layout()
plt.savefig(str(DATA_DIR.parent / 'reports' / 'eda_attack_distribution.png'), dpi=120)
plt.show()

## 3. Pipeline tiền xử lý

3 cột phân loại (`protocol_type`, `service` — ~70 mức, `flag`) → OneHot. Các cột số → StandardScaler
(cây quyết định/AdaBoost không *bắt buộc* cần scale, nhưng vẫn chuẩn hoá theo đúng yêu cầu của bài
để pipeline nhất quán và không ảnh hưởng xấu tới các thuật toán khác dùng để so sánh ở mục 8).
`handle_unknown='ignore'` cho OneHot vì `service` trong test có thể có giá trị hiếm không xuất hiện
ở train.

In [ ]:
y_train_full = train_df.pop('attack')
X_train_full = train_df

y_test = test_df.pop('attack')
X_test = test_df

cat_cols = ['protocol_type', 'service', 'flag']
num_cols = [c for c in X_train_full.columns if c not in cat_cols]

pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols),
])


def make_pipe(model):
    return Pipeline([('pre', pre), ('clf', model)])

## 4. Tách validation từ TRAIN (không đụng tới `KDDTest+.arff`)

Toàn bộ việc chọn siêu tham số, vẽ đường F1 theo n_estimators, và thí nghiệm nhiễu nhãn (mục 5-8)
đều dùng `X_tr`/`X_val` dưới đây. `X_test`/`y_test` (NSL-KDD gốc) chỉ xuất hiện lại ở mục 9.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=RANDOM_STATE)
print('X_tr:', X_tr.shape, '| X_val:', X_val.shape)

## 5. Baseline & AdaBoost (300 stump) so với 1 stump đơn lẻ

Không có bước tuning nào ở bước baseline nên đánh giá trên `X_val` là đủ minh hoạ; không cần chạm
tới test. Minh chứng trực tiếp cho sức mạnh của boosting: nhiều weak learner yếu (stump, depth=1)
cộng lại mạnh hơn hẳn một stump đơn lẻ.

In [ ]:
def eval_on_val(pipe, X_val, y_val):
    pred = pipe.predict(X_val)
    return {
        'accuracy': round(accuracy_score(y_val, pred), 3),
        'f1': round(f1_score(y_val, pred), 3),
        'precision': round(precision_score(y_val, pred, zero_division=0), 3),
        'recall': round(recall_score(y_val, pred, zero_division=0), 3),
    }


rows = []
for name, model in [
    ('Dummy (most_frequent)', DummyClassifier(strategy='most_frequent')),
    ('1 Stump (depth=1)', DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)),
]:
    pipe = make_pipe(model).fit(X_tr, y_tr)
    rows.append({'model': name, **eval_on_val(pipe, X_val, y_val)})

ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
    n_estimators=300, learning_rate=0.5, random_state=RANDOM_STATE,
)
ada_pipe = make_pipe(ada)
t0 = time.time()
ada_pipe.fit(X_tr, y_tr)
ada_train_time = time.time() - t0
rows.append({'model': 'AdaBoost (300 stump)', **eval_on_val(ada_pipe, X_val, y_val),
             'train_time_s': round(ada_train_time, 2)})

pd.DataFrame(rows)

## 6. Đường Accuracy / F1 theo số vòng lặp (n_estimators = 1..300)

Dùng `staged_predict` trên **validation**, không phải test, để chẩn đoán mà không rò rỉ.

In [ ]:
pre_fitted = ada_pipe.named_steps['pre']
Xval_t = pre_fitted.transform(X_val)
fitted_ada = ada_pipe.named_steps['clf']

f1_by_round, acc_by_round = [], []
for pred_stage in fitted_ada.staged_predict(Xval_t):
    f1_by_round.append(f1_score(y_val, pred_stage))
    acc_by_round.append(accuracy_score(y_val, pred_stage))

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(f1_by_round) + 1), f1_by_round, label='F1 (validation)')
plt.plot(range(1, len(acc_by_round) + 1), acc_by_round, label='Accuracy (validation)')
plt.xlabel('n_estimators'); plt.ylabel('Diem'); plt.legend()
plt.title('F1 / Accuracy theo so vong AdaBoost (do tren validation, khong phai test)')
plt.tight_layout()
plt.savefig(str(DATA_DIR.parent / 'reports' / 'f1_theo_vong_lap.png'), dpi=120)
plt.show()

print(f'F1 cao nhat: {max(f1_by_round):.3f} tai vong {int(np.argmax(f1_by_round)) + 1}')

## 7. ⭐ Thí nghiệm nhiễu nhãn: đảo ngẫu nhiên 5% nhãn train

AdaBoost tăng trọng số các mẫu bị phân sai liên tục — nếu một mẫu bị gán nhãn sai, nó sẽ luôn bị
"phân sai" theo nhãn (sai) đó, trọng số tăng gần như không giới hạn, khiến model dồn sức học đúng
điểm rác đó, kéo lệch toàn bộ ranh giới quyết định. Random Forest (bagging, biểu quyết đa số, không
tăng trọng số mẫu) được kỳ vọng ít nhạy hơn nhiều.

**Chỉ đảo nhãn trong `X_tr`/`y_tr` (phần huấn luyện)** — `X_val`/`y_val` giữ nguyên nhãn sạch để đo
mức độ mô hình bị kéo lệch một cách khách quan.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_flip = int(0.05 * len(y_tr))
flip_idx = rng.choice(y_tr.index, size=n_flip, replace=False)

y_tr_noisy = y_tr.copy()
y_tr_noisy.loc[flip_idx] = 1 - y_tr_noisy.loc[flip_idx]
print(f'Da dao nhan {n_flip}/{len(y_tr)} mau ({n_flip/len(y_tr):.1%}) trong tap train.')

noise_rows = []
for name, model in [
    ('AdaBoost', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
                                     n_estimators=300, learning_rate=0.5, random_state=RANDOM_STATE)),
    ('RandomForest', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
]:
    pipe_clean = make_pipe(model).fit(X_tr, y_tr)
    f1_clean = f1_score(y_val, pipe_clean.predict(X_val))

    pipe_noisy = make_pipe(model).fit(X_tr, y_tr_noisy)
    f1_noisy = f1_score(y_val, pipe_noisy.predict(X_val))

    noise_rows.append({
        'model': name,
        'f1_nhan_sach': round(f1_clean, 3),
        'f1_nhan_nhieu_5pct': round(f1_noisy, 3),
        'sut_giam': round(f1_clean - f1_noisy, 3),
        'sut_giam_pct': round((f1_clean - f1_noisy) / f1_clean * 100, 1),
    })

noise_df = pd.DataFrame(noise_rows)
noise_df.to_csv(str(DATA_DIR.parent / 'reports' / 'thi_nghiem_nhieu.csv'), index=False)
noise_df

In [ ]:
plt.figure(figsize=(6, 4))
x = np.arange(len(noise_df))
plt.bar(x - 0.2, noise_df['f1_nhan_sach'], width=0.4, label='Nhan sach')
plt.bar(x + 0.2, noise_df['f1_nhan_nhieu_5pct'], width=0.4, label='Nhan nhieu 5%')
plt.xticks(x, noise_df['model'])
plt.ylabel('F1 (validation)')
plt.title('Tac dong cua nhieu nhan: AdaBoost vs Random Forest')
plt.legend()
plt.tight_layout()
plt.savefig(str(DATA_DIR.parent / 'reports' / 'thi_nghiem_nhieu.png'), dpi=120)
plt.show()

print('Dien giai: neu AdaBoost sut F1 nhieu hon Random Forest -> xac nhan dung ly thuyet '
      '("AdaBoost rat nhay voi nhan sai vi co che tang trong so mau phan sai lien tuc").')

## 8. So sánh AdaBoost vs Gradient Boosting vs Random Forest (trên validation)

Ba mô hình dùng siêu tham số hợp lý mặc định/đã biết từ TT-03, TT-07, TT-09 — không có bước dò
thêm nào trên test ở đây.

In [ ]:
models_compare = {
    'AdaBoost': AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
                                    n_estimators=300, learning_rate=0.5, random_state=RANDOM_STATE),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=300, learning_rate=0.1, max_depth=3,
                                                     subsample=0.8, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
}

compare_rows = []
for name, model in models_compare.items():
    pipe = make_pipe(model)
    t0 = time.time(); pipe.fit(X_tr, y_tr); dt = time.time() - t0
    pred = pipe.predict(X_val)
    compare_rows.append({
        'model': name,
        'f1_val': round(f1_score(y_val, pred), 3),
        'accuracy_val': round(accuracy_score(y_val, pred), 3),
        'precision_val': round(precision_score(y_val, pred, zero_division=0), 3),
        'recall_val': round(recall_score(y_val, pred, zero_division=0), 3),
        'train_time_s': round(dt, 2),
    })
compare_df = pd.DataFrame(compare_rows)
compare_df.to_csv(str(DATA_DIR.parent / 'reports' / 'so_sanh_ensemble.csv'), index=False)
compare_df

In [ ]:
compare_df.set_index('model')[['f1_val', 'accuracy_val']].plot(kind='bar', figsize=(6, 4))
plt.ylabel('Diem (validation)'); plt.title('So sanh AdaBoost vs Gradient Boosting vs Random Forest')
plt.tight_layout()
plt.savefig(str(DATA_DIR.parent / 'reports' / 'so_sanh_ensemble.png'), dpi=120)
plt.show()

## 9. Đánh giá cuối cùng trên `KDDTest+.arff` (tập test GỐC, có tấn công lạ)

**Chỉ dùng test đúng một lần**, sau khi mọi lựa chọn đã chốt ở các mục trên. Mô hình cuối được
refit trên **toàn bộ** `X_train_full`/`y_train_full` (gồm cả phần trước đó dùng làm validation) để
tận dụng hết dữ liệu train trước khi đánh giá trên test — đây là bước áp dụng, không phải bước chọn
mô hình, nên không gây rò rỉ.

In [ ]:
ada_final = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
    n_estimators=300, learning_rate=0.5, random_state=RANDOM_STATE,
)
ada_final_pipe = make_pipe(ada_final).fit(X_train_full, y_train_full)

pred_test = ada_final_pipe.predict(X_test)
f1_test = f1_score(y_test, pred_test)
acc_test = accuracy_score(y_test, pred_test)

f1_val_adaboost = compare_df.loc[compare_df.model == 'AdaBoost', 'f1_val'].values[0]
print(f"F1 tren validation (muc 8, cung cau hinh): {f1_val_adaboost}")
print(f"F1 tren KDDTest+.arff (tan cong la):        {f1_test:.3f}")
print(f"Chenh lech: {f1_val_adaboost - f1_test:.3f}")
print()
print("Dien giai: theo README, chenh lech diem CV/validation cao hon han test goc LA KET QUA DUNG -")
print("test NSL-KDD co y chua loai tan cong khong co trong train (mo phong zero-day). Ban ARFF")
print("khong cho biet CU THE tan cong nao la moi (vi nhan da nhi phan hoa), nhung co che gay ra")
print("chenh lech van khong doi: mo hinh khong nhan dien duoc kieu tan cong no chua tung thay.")

## 10. Ma trận nhầm lẫn & ước tính báo động giả mỗi ngày

Giả định quy mô SOC (cần thay bằng số thật của tổ chức): **N_CONN_PER_DAY** kết nối/ngày, tỉ lệ
`normal`/`attack` gần đúng theo phân bố của `KDDTest+.arff`. Số báo động giả/ngày = FP_rate ×
(số kết nối normal thực tế mỗi ngày) — đây là con số quyết định trực tiếp mức độ "alert fatigue"
của nhân viên SOC.

In [ ]:
cm = confusion_matrix(y_test, pred_test)
tn, fp, fn, tp = cm.ravel()
fp_rate = fp / (fp + tn)   # ti le bao dong gia tren cac ket noi THUC SU binh thuong
fn_rate = fn / (fn + tp)   # ti le bo sot tren cac ket noi THUC SU la tan cong

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['normal', 'attack']).plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Ma tran nham lan tren KDDTest+.arff')
plt.tight_layout()
plt.savefig(str(DATA_DIR.parent / 'reports' / 'confusion_matrix_test.png'), dpi=120)
plt.show()

N_CONN_PER_DAY = 1_000_000   # THAY bang so ket noi/ngay that cua SOC dang giam sat
normal_share_test = (y_test == 0).mean()
est_normal_per_day = N_CONN_PER_DAY * normal_share_test
est_false_alarms_per_day = est_normal_per_day * fp_rate

print(f"FP rate (bao dong gia tren ket noi normal that): {fp_rate:.3%}")
print(f"FN rate (bo sot tren ket noi attack that):        {fn_rate:.3%}")
print(f"Voi gia dinh {N_CONN_PER_DAY:,} ket noi/ngay, ~{normal_share_test:.0%} la normal:")
print(f"  -> Uoc tinh ~{est_false_alarms_per_day:,.0f} bao dong gia/ngay")
print(f"  -> Uoc tinh bo sot ~{fn_rate:.1%} cac cuoc tan cong that")
print("N_CONN_PER_DAY la so gia dinh minh hoa -- can thay bang luu luong that cua SOC.")

## 11. Lưu model & tổng kết

In [ ]:
MODEL_PATH = DATA_DIR.parent / 'models' / 'adaboost.joblib'
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(ada_final_pipe, MODEL_PATH)
print('Da luu model tai (duong dan tuong doi):', MODEL_PATH)

final_summary = pd.DataFrame([{
    'model': 'AdaBoost (final, fit tren toan bo train)',
    'f1_validation_tuong_duong': f1_val_adaboost,
    'f1_test_goc_tan_cong_la': round(f1_test, 3),
    'accuracy_test_goc': round(acc_test, 3),
    'fp_rate_test': round(fp_rate, 4),
    'fn_rate_test': round(fn_rate, 4),
}])
final_summary.to_csv(str(DATA_DIR.parent / 'reports' / 'final_summary.csv'), index=False)
final_summary